# Smoke test for `new_corr`

This notebook creates small synthetic transcriptomics/proteomics tables and runs:
- `compute_correlation_for_protein`
- `run_correlation_group`

Outputs are written into `./smoke_out/`.


In [1]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd

from corr_functions import compute_correlation_for_protein, run_correlation_group

np.random.seed(42)

out_dir = Path("smoke_out")
out_dir.mkdir(exist_ok=True)


In [2]:
# Build synthetic data
n_samples = 20
n_genes = 50
n_prots = 5

samples = [f"S{i:02d}" for i in range(n_samples)]
genes = [f"G{i:03d}" for i in range(n_genes)]
prots = [f"P{i:03d}" for i in range(n_prots)]

# transcriptomics: random noise
X = pd.DataFrame(np.random.normal(size=(n_samples, n_genes)), index=samples, columns=genes)

# make gene G000 strongly related to protein P000
signal = np.random.normal(size=n_samples)
X["G000"] = signal + np.random.normal(scale=0.2, size=n_samples)

# proteomics: random + one correlated
Y = pd.DataFrame(np.random.normal(size=(n_samples, n_prots)), index=samples, columns=prots)
Y["P000"] = signal + np.random.normal(scale=0.2, size=n_samples)

X.index.name = "samples"
Y.index.name = "samples"
X.columns.name = "genes"
Y.columns.name = "proteins"

X.head(), Y.head()


(genes        G000      G001      G002      G003      G004      G005      G006  \
 samples                                                                         
 S00      1.503923 -0.138264  0.647689  1.523030 -0.234153 -0.234137  1.579213   
 S01      0.809894 -0.385082 -0.676922  0.611676  1.031000  0.931280 -0.839218   
 S02      0.054759 -0.420645 -0.342715 -0.802277 -0.161286  0.404051  1.886186   
 S03     -0.218483  0.346448 -0.680025  0.232254  0.293072 -0.714351  1.865775   
 S04      1.043732  0.560785  1.083051  1.053802 -1.377669 -0.937825  0.515035   
 
 genes        G007      G008      G009  ...      G040      G041      G042  \
 samples                                ...                                 
 S00      0.767435 -0.469474  0.542560  ...  0.738467  0.171368 -0.115648   
 S01     -0.309212  0.331263  0.975545  ...  0.097078  0.968645 -0.702053   
 S02      0.174578  0.257550 -0.074446  ...  0.227460  1.307143 -1.607483   
 S03      0.473833 -1.191303  0.656554 

In [3]:
# Single protein correlation
s = compute_correlation_for_protein(
    transcriptomics=X,
    proteomics=Y,
    protein_name="P000",
    bootstrap=True,
    n_bootstraps=200,
    random_state=123,
)

# show top non-zero correlations
nz = s[s != 0.0].sort_values(key=lambda v: v.abs(), ascending=False)
nz.head(10)


genes
G000    0.945318
G029   -0.518443
G044   -0.517623
G032    0.470348
G005    0.443811
G009    0.385512
G012   -0.289049
dtype: float64

In [4]:
# Group run (write per-protein CSVs to avoid parquet engine issues)
batch = run_correlation_group(
    proteins=["P000", "P001", "P002"],
    transcriptomics=X,
    proteomics=Y,
    bootstrap=True,
    n_bootstraps=200,
    random_state=42,
    output_dir=out_dir,
    per_protein_format="csv",
)

batch


,protein,n_sig,corr_json
0,P000,6,"{""G000"": 0.9463557538780037, ""G005"": 0.4257163..."
1,P001,4,"{""G000"": -0.5084297230910049, ""G002"": -0.66915..."
2,P002,2,"{""G030"": -0.42826925479025685, ""G033"": -0.6268..."


In [5]:
# Verify per-protein files exist
per_dir = out_dir / "per_protein"
files = sorted(per_dir.glob("*_corr.csv"))
files


[WindowsPath('smoke_out/per_protein/P000_corr.csv'),
 WindowsPath('smoke_out/per_protein/P001_corr.csv'),
 WindowsPath('smoke_out/per_protein/P002_corr.csv')]

In [6]:
# Inspect one output file
df_one = pd.read_csv(files[0], index_col=0)
df_one.iloc[:, :10]


,G000,G001,G002,G003,G004,G005,G006,G007,G008,G009
protein,,,,,,,,,,
P000,0.946356,0.0,0.0,0.0,0.0,0.425716,0.0,0.0,0.0,0.411544


In [7]:
# Decode corr_json for the first protein in the batch
row0 = batch.iloc[0]
corr_dict = json.loads(row0["corr_json"])
list(corr_dict.items())[:10], row0["n_sig"]


([('G000', 0.9463557538780037),
  ('G005', 0.42571636437831334),
  ('G009', 0.4115438315904104),
  ('G029', -0.5032467143070998),
  ('G032', 0.4430961111635173),
  ('G044', -0.5029957323752939)],
 6)

## Real data test

In [10]:
# --- Real-data smoke test (UPDATED: assumes data/ is INSIDE new_corr/) ---
from pathlib import Path
import json

import numpy as np
import pandas as pd
from joblib import load

from corr_functions import compute_correlation_for_protein, run_correlation_group

# Resolve new_corr root whether notebook runs from project root or from ./new_corr
cwd = Path.cwd()

if (cwd / "data").exists():
    # running from inside new_corr (recommended)
    new_corr_root = cwd
elif (cwd / "new_corr" / "data").exists():
    # running from project root
    new_corr_root = cwd / "new_corr"
else:
    # fallback: walk up to find a folder that contains new_corr/data
    new_corr_root = None
    for p in [cwd] + list(cwd.parents):
        if (p / "new_corr" / "data").exists():
            new_corr_root = p / "new_corr"
            break
    if new_corr_root is None:
        raise FileNotFoundError("Could not find 'new_corr/data' from current working directory.")

data_dir = new_corr_root / "data"

trans_path = data_dir / "transcriptomics_filtered.joblib"
prot_path  = data_dir / "proteomics_matched.joblib"
plist_path = data_dir / "protein_list.txt"

assert trans_path.exists(), f"Missing: {trans_path}"
assert prot_path.exists(),  f"Missing: {prot_path}"

print("Loading joblibs...")
X = load(str(trans_path))  # transcriptomics (samples x genes)
Y = load(str(prot_path))   # proteomics (samples x proteins)

# Make sure index/columns naming is consistent (optional but nice)
X.index.name = "samples"
Y.index.name = "samples"
X.columns.name = "genes"
Y.columns.name = "proteins"

print("Shapes:", "X=", X.shape, "Y=", Y.shape)

# Pick a small subset of genes for a quick smoke test (optional, but recommended)
N_GENES = 500
rng = np.random.default_rng(42)
if X.shape[1] > N_GENES:
    gene_subset = rng.choice(X.columns.to_numpy(), size=N_GENES, replace=False)
    X_sm = X.loc[:, gene_subset]
else:
    X_sm = X

# Choose a few proteins (prefer protein_list.txt if present)
def read_protein_list(path: Path):
    return [ln.strip() for ln in path.read_text().splitlines() if ln.strip() and not ln.strip().startswith("#")]

if plist_path.exists():
    candidates = read_protein_list(plist_path)
else:
    candidates = list(Y.columns)

proteins = [p for p in candidates if p in Y.columns][:3]
if len(proteins) == 0:
    raise ValueError("No proteins found to test (protein_list empty or none match Y.columns).")

print("Using proteins:", proteins)
print("Using transcriptomics subset:", X_sm.shape)

# ---- 1) Single-protein function test ----
prot0 = proteins[0]
s = compute_correlation_for_protein(
    transcriptomics=X_sm,
    proteomics=Y,
    protein_name=prot0,
    bootstrap=True,
    n_bootstraps=200,
    random_state=123,
)

nz = s[s != 0.0].sort_values(key=lambda v: v.abs(), ascending=False)
print(f"\nSingle protein: {prot0} | non-zero correlations: {len(nz)}")
display(nz.head(15))

# ---- 2) Group function test (NO saving; output_dir=None) ----
batch = run_correlation_group(
    proteins=proteins,
    transcriptomics=X_sm,
    proteomics=Y,
    bootstrap=True,
    n_bootstraps=200,
    random_state=42,
    output_dir=None,  # IMPORTANT: no files written
)

print("\nBatch summary:")
display(batch)

# Inspect decoded corr_json for the first protein
corr_dict0 = json.loads(batch.loc[0, "corr_json"])
print(f"\nDecoded corr_json for {batch.loc[0, 'protein']} (showing up to 10 items):")
display(list(corr_dict0.items())[:10])

# ---- (Optional) If we want to write outputs like the pipeline does ----
# out_dir = data_dir / "results"
# out_dir.mkdir(parents=True, exist_ok=True)
# batch_saved = run_correlation_group(
#     proteins=proteins,
#     transcriptomics=X_sm,
#     proteomics=Y,
#     bootstrap=True,
#     n_bootstraps=200,
#     random_state=42,
#     output_dir=out_dir,
#     per_protein_format="parquet",
# )
# display(batch_saved)
# print("Wrote per-protein files under:", out_dir / "per_protein")


Loading joblibs...
Shapes: X= (369, 22614) Y= (369, 5113)
Using proteins: ['A2M', 'AAAS', 'AACS']
Using transcriptomics subset: (369, 500)

Single protein: A2M | non-zero correlations: 214


genes
SAMD10       -0.378156
AC244517.6    0.342777
CNN3          0.334368
CNTNAP1       0.327534
MRC2          0.309785
GSTA4         0.306787
AC145124.1    0.294756
ELMOD1        0.292397
STX2          0.288459
EVI5          0.285670
PYGO1         0.284673
HOXC8         0.283563
DCHS1         0.281810
FAM151B       0.281242
DENND2B       0.280964
dtype: float64


Batch summary:


,protein,n_sig,corr_json
0,A2M,221,"{""GNPTAB"": 0.09086979534141801, ""GAS6-AS1"": -0..."
1,AAAS,114,"{""TOMM7"": -0.187403157008741, ""AKT1"": -0.11661..."
2,AACS,90,"{""TOMM7"": 0.1394604236507324, ""SLC35A1"": 0.156..."



Decoded corr_json for A2M (showing up to 10 items):


[('GNPTAB', 0.09086979534141801),
 ('GAS6-AS1', -0.10456507667130577),
 ('CPEB1', 0.2578521569606532),
 ('RNF144A', 0.16087969607658786),
 ('DRG1', 0.20124523851334594),
 ('CAPN2', 0.11546654055918928),
 ('HOXC8', 0.28489349415435816),
 ('SCAF1', 0.14732963566847881),
 ('AC007191.1', -0.13014188608955427),
 ('NLGN2', 0.2848070554617745)]